In [ ]:
import numpy as np
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from dim_reduction_utils import load_imbalanced_cryptic_and_regular_data
from constants import CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET, EMB_SPACES, CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET, EMBEDDINGS_PATH, IMG_OUTPUT_PATH

MODELS_PATH = f'{EMBEDDINGS_PATH}/../models'


In [ ]:

def process_emma_data(train_emma, embedding_space):
    X = train_emma.emb[embedding_space[0]]['emb']
    y = np.zeros((len(train_emma.metadata))) # 3 classes: Non-binding, Regular, Cryptic
    y += (train_emma.metadata['binding_site'] == 'BINDING').to_numpy().astype(int) # Regular binding = 1
    y += (train_emma.metadata['binding_site'] == 'CRYPTIC-BINDING').to_numpy().astype(int) * 2 # Cryptic binding = 2

    X, y = shuffle(X, y, random_state=42)
    return X, y

def perform_linear_probing():
    for embedding_space in EMB_SPACES:
        print(f'Performing linear probing for {embedding_space[0]}...')
        train_emma = load_imbalanced_cryptic_and_regular_data(embedding_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET])
        test_emma = load_imbalanced_cryptic_and_regular_data(embedding_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET])
        X_train, y_train = process_emma_data(train_emma, embedding_space)
        X_test, y_test = process_emma_data(test_emma, embedding_space)
        # print("Creating scaler and scaling data...")
        # scaler = StandardScaler()
        # X_train_scaled = scaler.fit_transform(X_train)
        # X_test_scaled = scaler.transform(X_test)
        X_test_scaled = X_test
        X_train_scaled = X_train

        # 4. Multiclass Linear Probe
        # multi_class='multinomial' uses the cross-entropy loss (softmax)
        print("Training linear probe...")
        probe = LogisticRegression(
            max_iter=2000, 
            class_weight='balanced', 
            random_state=42
        )

        probe.fit(X_train_scaled, y_train)

        # 5. Predictions
        print("Making predictions...")
        y_pred = probe.predict(X_test_scaled)
        y_prob = probe.predict_proba(X_test_scaled) # Returns [n_samples, 3]

        # 6. Evaluation
        print(f"--- 3-Class Linear Probe Results: {embedding_space[0]} ---")
        # Use 'macro' average to give equal weight to the 'Cryptic' class
        print(classification_report(y_test, y_pred, target_names=['Non-binding', 'Regular', 'Cryptic']))

        # Multiclass ROC-AUC (One-vs-Rest)
        auc_ovr = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
        print(f"Multiclass ROC-AUC (OVR): {auc_ovr:.4f}")

        # 7. Static Visualization: Confusion Matrix
        cm = confusion_matrix(y_test, y_pred, normalize='true') # Normalized by row (true labels)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', 
                    xticklabels=['Non-binding', 'Regular', 'Cryptic'], 
                    yticklabels=['Non-binding', 'Regular', 'Cryptic'])
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title('Normalized Confusion Matrix: Binding Site Types')
        plt.show()

perform_linear_probing()

Performing linear probing for ESM2...
2042764 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Training linear probe...


: 